# 4.1 — Cadastro comercial de clientes (KNVV)

- **Propósito:** Consolidar os atributos comerciais de clientes provenientes do SAP KNVV.
- **Entrada:** Arquivos XLS/XLSX do volume `dm_customers/knvv_sap`
- **Saída:** `parts_hdbk_sandbox.dm_customers.knvv_sap`
- **Chave:** Cliente + Organização de vendas + Canal + Setor · **Carga:** Completa

In [0]:
import pandas as pd
import glob
from pyspark.sql import functions as F

# Bibliotecas para leitura de arquivos Excel e manipulação de dados

In [0]:
# Instalar bibliotecas para leitura de arquivos XLS/XLSX
%pip install openpyxl xlrd --quiet

In [0]:
# Caminho do volume com os arquivos XLS
VOLUME_PATH = "/Volumes/parts_hdbk_sandbox/dm_customers/knvv_sap"

# Ler todos os arquivos XLS do volume usando pandas e converter para Spark DF
xls_files = glob.glob(f"{VOLUME_PATH}/*.xls*")
print(f"Arquivos encontrados: {len(xls_files)}")
for f in xls_files:
    print(f"  - {f}")

# Concatenar todos os arquivos em um único pandas DataFrame
pdf_list = []
for f in xls_files:
    pdf_part = pd.read_excel(f, dtype=str)
    pdf_list.append(pdf_part)
    print(f"  {f}: {len(pdf_part)} linhas")

pdf = pd.concat(pdf_list, ignore_index=True)
print(f"\nTotal de linhas lidas: {len(pdf)}")

# Converter para Spark DataFrame
df_knvv = spark.createDataFrame(pdf)

# Arquivos carregados e convertidos para Spark DataFrame

In [0]:
import re
import unicodedata


def normalize_col_name(name: str) -> str:
    """Normaliza nome de coluna: remove acentos, lowercase, troca caracteres
    especiais por underscore e remove underscores extras."""
    # Remover acentos
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    # Lowercase
    name = name.lower()
    # Substituir caracteres não-alfanuméricos por underscore
    name = re.sub(r"[^a-z0-9]+", "_", name)
    # Remover underscores no início/fim
    name = name.strip("_")
    return name


# Aplicar normalização a todas as colunas
original_cols = df_knvv.columns
new_cols = [normalize_col_name(c) for c in original_cols]

print("Mapeamento de colunas:")
for orig, new in zip(original_cols, new_cols):
    df_knvv = df_knvv.withColumnRenamed(orig, new)
    print(f"  {orig:15s} -> {new}")

# --- Transformações de negócio ---

# Remover zeros à esquerda da coluna cliente
df_knvv = df_knvv.withColumn(
    "cliente",
    F.regexp_replace(F.col("cliente"), r"^0+", "")
)

# Preencher cen nulo/vazio com centro padrão por orgv
# orgv 0200 -> 0203 | orgv 0500 -> 0503
df_knvv = df_knvv.withColumn(
    "cen",
    F.when(
        (F.col("cen").isNull()) | (F.trim(F.col("cen")) == ""),
        F.when(F.col("orgv") == "0200", F.lit("0203"))
         .when(F.col("orgv") == "0500", F.lit("0503"))
    ).otherwise(F.col("cen"))
)

print("\nTransformações aplicadas:")
print("  - Colunas normalizadas (lowercase, sem caracteres especiais).")
print("  - Zeros à esquerda removidos da coluna 'cliente'.")
print("  - 'cen' nulo/vazio preenchido com centro padrão (0203 ou 0503).")
print("\nSchema final:")
df_knvv.printSchema()
print("Amostra:")
display(df_knvv.limit(10))

In [0]:
DEST_TABLE = "parts_hdbk_sandbox.dm_customers.knvv_sap"

# Dropar tabela antiga se existir
spark.sql(f"DROP TABLE IF EXISTS {DEST_TABLE}")

# Criar tabela Delta
df_knvv.write.saveAsTable(DEST_TABLE)

# Tabela criada com sucesso

In [0]:
%sql
-- Definir colunas NOT NULL (requisito para PK)
ALTER TABLE parts_hdbk_sandbox.dm_customers.knvv_sap
ALTER COLUMN cliente SET NOT NULL;

ALTER TABLE parts_hdbk_sandbox.dm_customers.knvv_sap
ALTER COLUMN orgv SET NOT NULL;

ALTER TABLE parts_hdbk_sandbox.dm_customers.knvv_sap
ALTER COLUMN cdst SET NOT NULL;

ALTER TABLE parts_hdbk_sandbox.dm_customers.knvv_sap
ALTER COLUMN sa SET NOT NULL;

-- Adicionar constraint de chave primaria composta
ALTER TABLE parts_hdbk_sandbox.dm_customers.knvv_sap
ADD CONSTRAINT pk_knvv_sap PRIMARY KEY (cliente, orgv, cdst, sa);

In [0]:
# Aplicar comentários e tags do Unity Catalog na tabela e colunas
DEST_TABLE = "parts_hdbk_sandbox.dm_customers.knvv_sap"

# Comentário da tabela
TABLE_COMMENT = """
Tabela de area de vendas do cliente (KNVV SAP) com informações do Centro que atende o cliente.
Ingestao a partir de arquivos XLS do volume knvv_sap.

Chave Primaria: Cliente + OrgV + CDst + SA
Atualizacao: Carga manual via volume.

Relacionamentos:
  - Cliente -> codigo do cliente SAP
  - OrgV -> organizacao de vendas
  - CDst -> canal de distribuicao
  - SA -> setor de atividade
"""

spark.sql(f"""
    COMMENT ON TABLE {DEST_TABLE} IS '{TABLE_COMMENT.replace(chr(39), chr(39)+chr(39))}'
""")

# Tags do Unity Catalog
spark.sql(f"""
    ALTER TABLE {DEST_TABLE} SET TAGS (
        'domain' = 'customers',
        'layer' = 'refined',
        'source' = 'sap',
        'source_table' = 'KNVV',
        'data_classification' = 'internal'
    )
""")

# Propriedades customizadas
spark.sql(f"""
    ALTER TABLE {DEST_TABLE} SET TBLPROPERTIES (
        'business_owner' = 'Demand Planning',
        'technical_owner' = 'Andre Causs',
        'data_domain' = 'Customer',
        'source_system' = 'SAP',
        'source_path' = '/Volumes/parts_hdbk_sandbox/dm_customers/knvv_sap',
        'refresh_frequency' = 'manual_volume_upload',
        'primary_key' = 'cliente, orgv, cdst, sa'
    )
""")

# Comentários nas colunas
COLUMN_COMMENTS = {
    "cliente": "Codigo do cliente SAP (sem zeros a esquerda). Parte da chave primaria.",
    "orgv": "Organizacao de vendas SAP (ex: 0200=2W, 0500=4W). Parte da chave primaria.",
    "cdst": "Canal de distribuicao (ex: 01=Revenda). Parte da chave primaria.",
    "sa": "Setor de atividade (ex: 04=Pecas, 07=Pos-Venda). Parte da chave primaria.",
    "cen": "Centro que atende o cliente. Quando nulo na origem, preenchido com padrao (0203 para orgv 0200, 0503 para orgv 0500).",
}

for column_name, comment in COLUMN_COMMENTS.items():
    escaped_comment = comment.replace("'", "''")
    spark.sql(f"COMMENT ON COLUMN {DEST_TABLE}.{column_name} IS '{escaped_comment}'")

print(f"Metadados completos aplicados a tabela {DEST_TABLE}")